# ANALYSIS OF APARTMENT PROPERTIES

## Analyst: Aaron Hardy
## Date: June 7, 2026
---

## Goal
1. Find the markets where rents are rising the fastest and rising the slowest
<!-- 2. Find the markets where renter demand is rising the fastest and rising the slowest
3. Find which markets have the highest SFR sales price per square foot
4. Find markets where apartment properties are likely to rise due to potential rental rate growth, growing renter demand and higher construction median sale price per square foot -->


## Data: Zillow
### ZORI
- URL: https://www.zillow.com/research/data/?msockid=09f1e4300f88632836b5f3550e666237
- Zillow Observed Rent Index (ZORI): A smoothed measure of the typical observed market rate rent across a given region. ZORI is a repeat-rent index that is weighted to the rental housing stock to ensure representativeness across the entire market, not just those homes currently listed for-rent. The index is dollar-denominated by computing the mean of listed rents that fall into the 35th to 65th percentile range for all homes and apartments in a given region, which is weighted to reflect the rental housing stock.
ZORI is created for three different categories: All homes, Single Family Residences, and Multi-Family Residences. For more detailed information, you can refer to the ZORI methodology.

### Zillow Smoothing and Seasonality-Adjustments
Once the index is computed, it is smoothed using a three-month simple moving average. A seasonally adjusted series is available and calculated using a proprietary algorithm that is more robust to outlier events such as the 2020 pandemic.

<!-- ### ZORDI
https://www.zillow.com/research/data/?msockid=09f1e4300f88632836b5f3550e666237
Zillow Observed Renter Demand Index (ZORDI): A measure of the typical observed rental market engagement across a region. ZORDI tracks engagement on Zillow’s rental listings to proxy changes in rental demand. The metric is smoothed to remove volatility.
ZORDI is created for different categories including All homes, Single Family Residences, Condo and Multi-Family Residences at national and MSA levels.

### New Construction Median Sale Price Per Sqft
https://www.zillow.com/research/data/?msockid=09f1e4300f88632836b5f3550e666237
New Construction Median Sale Price Per Sqft: The median sale price divided by square footage calculated over all new construction homes across various geographies were sold during the month. -->

### City Populations
https://www.census.gov/data/tables/time-series/demo/popest/2020s-total-cities-and-towns.html

In [602]:

# ===============================
# STEP 1: LOAD DATA
# ===============================

import pandas as pd
import numpy as np
import duckdb

# small = pd.read_csv("data/raw/apartments_for_rent_classified_10K.csv")
# large = pd.read_csv("data/raw/apartments_for_rent_classified_10K.csv")
# Metro ZORI, multi-family, seasonally-adjusted, smoothed, monthly
zori_raw = pd.read_csv("data/raw/zillow/Metro_zori_uc_mfr_sm_month.csv", index_col=None)
#
zordi_raw_all = pd.read_csv("data/raw/zillow/Metro_zordi_uc_sfrcondomfr_month.csv")
#
zordi_raw_multifam = pd.read_csv("data/raw/zillow/Metro_zordi_uc_mfr_month.csv")
#
price_per_sqf_sfr_condo_raw = pd.read_csv("data/raw/zillow/Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv")
# Census population estimates (released May 2026)
pop_raw = pd.read_csv("data/raw/census/cbsa-est2025-alldata.csv", encoding="latin1")

# print(zori.head(3))
# print(zordi_all.head(3))
# print(zordi_multifam.head(3))
# print(pop.head(3))


## 1. ZORI (Zillow Observed Rent Index)

No. of cities: 932


In [603]:
# ==========================
# STEP 2: Round index values
# ==========================
zori = zori_raw.round(0)

# =============================
# STEP 3: Pivot shape
# ==============================
zori = zori.melt(
    id_vars=["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"],
    var_name="Date",
    value_name="Zori",
)

# ============================
# STEP 4: Remove nulls
# ============================
zori.dropna(subset="Zori", inplace=True)
# print(zori[zori['Zori'].isna()])

# =============================
# STEP 5: Format Date
# =============================
# print(zori['Date'].dtype)
zori["Date"] = pd.to_datetime(zori["Date"], errors="coerce")

# print(zori['Date'].min())
# print(zori['Date'].max())
# print(zori['Date'].isna().sum())

# =============================
# STEP 6: Add CAGR
# =============================


def add_cagr(df, years):
    months = 12 * years
    df = df.sort_values(["RegionName", "Date"])

    df[f"CAGR_{years}yr"] = df.groupby("RegionName")["Zori"].transform(
        lambda x: ((x / x.shift(months)) ** (1 / years) - 1).round(3)
    )
    return df


zori = add_cagr(zori, 1)
zori = add_cagr(zori, 2)
zori = add_cagr(zori, 5)


def add_vol(df, years):
    months = 12 * years
    df["pct_chg"] = df.groupby("RegionName")["Zori"].pct_change()
    df = df.sort_values(["RegionName", "Date"])

    df[f"SD_{years}yr"] = df.groupby("RegionName")["pct_chg"].transform(
        lambda x: (x.rolling(months).std() * np.sqrt(12)).round(3)
    )
    return df


zori = add_vol(zori, 1)
zori = add_vol(zori, 2)
zori = add_vol(zori, 5)


# print(zori.dropna(subset=['CAGR_1yr', 'CAGR_2yr', 'CAGR_5yr']).head(10))

# =============================
# STEP _: Calculate growth rates
# =============================

# Calculate 3-month simple moving average of rental growth rates
# zori['SMA_Growth_3MO'] = (
#     zori.groupby('RegionName')['Growth_Rate'].transform(lambda x: x.rolling(window=3).mean())
# )

# =============================
# STEP 7: Remove regions with short time series
# =============================

region_counts = zori["RegionName"].value_counts()
sufficient_data_regions = region_counts[region_counts >= 5 * 12 + 2].index
zori = zori[zori["RegionName"].isin(sufficient_data_regions)]

zori.nlargest(10, "CAGR_5yr")
# Rank of the percent growth within each month (1 = highest growth rate)
# Growth rate is the percent change in ZORI from the previous month for each region
# ZORI smoothed is itself a 3-month moving average of rental rates

# =============================
# STEP 8: Remove missing value rows
# =============================

zori = zori[~zori["CAGR_5yr"].isna()]

# =============================
# STEP _: Calculate Ranks
# =============================
# zori['Growth_Rank_CAGR_1yr'] = zori.groupby('Date')['CAGR_1yr'].rank(ascending=False, method='dense')
# zori['Growth_Rank_CAGR_2yr'] = zori.groupby('Date')['CAGR_2yr'].rank(ascending=False, method='dense')
# zori['Growth_Rank_CAGR_5yr'] = zori.groupby('Date')['CAGR_5yr'].rank(ascending=False, method='dense')


# =============================
# STEP 10: Remove Outliers
# =============================

TUKEY_FACTOR = 1.5
Q1 = zori["CAGR_1yr"].quantile(0.25)
Q3 = zori["CAGR_1yr"].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - TUKEY_FACTOR * IQR
upper_limit = Q3 + TUKEY_FACTOR * IQR

# zori = zori.loc[(zori['CAGR_1yr'] < upper_limit) & zori['CAGR_1yr'] > lower_limit]
zori = zori.loc[zori["CAGR_1yr"].between(lower_limit, upper_limit)]


# =============================
# STEP 11: Subset the Data
# =============================
zori = (
    zori[
        [
            "RegionName",
            "Date",
            "Zori",
            "SizeRank",
            "CAGR_1yr",
            "CAGR_2yr",
            "CAGR_5yr",
            "SD_1yr",
            "SD_2yr",
            "SD_5yr",
        ]
    ]
    .sort_values(
        ["Date", "CAGR_1yr", "CAGR_2yr", "CAGR_5yr", "SD_1yr", "SD_2yr", "SD_5yr"],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ===========================================
# STEP _: Estimate missing population values
# ===========================================
# An attempt to interpolate the missing populations resulted in extreme values that were double their verified numbers via Google search


# =============================
# STEP 9: Add Population Data
# =============================

# Select the MSA data
pop = pop_raw.loc[
    pop_raw["LSAD"] == "Metropolitan Statistical Area", ["NAME", "POPESTIMATE2025"]
]

#     NAME              POPESTIMATE2025
# 0   Abilene, TX       185429
# 4   Akron, OH         701780

merged = (
    zori.merge(pop, left_on="RegionName", right_on="NAME", how="left")
    .drop(columns=["NAME"])
    .rename(columns={"POPESTIMATE2025": "Population 2025"})
    .sort_values(["Date", "CAGR_1yr"], ascending=False)
)

# merged['Population 2025'] = merged['Population 2025'].fillna('Not specified')


# =================================================
# STEP 11: Select largest 300 markets (by MSA size)
# =================================================
# print(merged['SizeRank'].min())
# print(merged['SizeRank'].max())


merged = merged[merged["SizeRank"] <= 200]


merged.head(10).style.set_caption("""
    <div style='font-size:18px; font-weight:600;'>Which cities are leading rental rate growth?</div>
    <div style='font-size:13px; color:#888; margin-bottom: 2px;'>Rank movements over the last 18 months</div>
    """).set_table_styles(
    [
        {
            "selector": "caption",
            "props": [("caption-side", "top"), ("text-align", "left")],
        }
    ]
).format(
    {
        "Date": lambda x: x.strftime("%b %Y"),
        "Zori": "{:,.0f}",
        "CAGR_1yr": "{:.1%}",
        "CAGR_2yr": "{:.1%}",
        "CAGR_5yr": "{:.1%}",
        "SD_1yr": "{:.1%}",
        "SD_2yr": "{:.1%}",
        "SD_5yr": "{:.1%}",
        "Population 2025": lambda x: "-" if pd.isna(x) else f"{x:,.0f}",
    }
).hide(
    axis="index"
)

RegionName,Date,Zori,SizeRank,CAGR_1yr,CAGR_2yr,CAGR_5yr,SD_1yr,SD_2yr,SD_5yr,Population 2025
"Cedar Rapids, IA",Apr 2026,"1,014",179,12.9%,8.5%,6.2%,3.1%,4.2%,3.7%,"279,285"
"Evansville, IN",Apr 2026,"1,001",165,12.0%,9.7%,9.9%,8.5%,6.2%,5.3%,"273,786"
"Binghamton, NY",Apr 2026,"1,213",195,8.9%,7.1%,9.6%,5.8%,5.1%,5.8%,"243,189"
"Reno, NV",Apr 2026,"1,783",113,8.9%,5.9%,4.3%,3.3%,3.2%,3.6%,"578,734"
"Gulfport, MS",Apr 2026,"1,206",133,8.3%,6.6%,8.2%,4.1%,4.3%,4.9%,-
"Salisbury, MD",Apr 2026,"1,767",134,6.5%,6.1%,6.5%,4.2%,3.6%,5.9%,"131,872"
"Urban Honolulu, HI",Apr 2026,"2,393",55,6.4%,5.4%,5.2%,3.1%,2.3%,2.6%,"988,703"
"San Francisco, CA",Apr 2026,"2,991",12,6.2%,4.9%,4.1%,1.5%,1.5%,2.2%,-
"Rockford, IL",Apr 2026,"1,057",153,6.0%,6.8%,7.2%,3.1%,3.1%,2.9%,"337,242"
"Virginia Beach, VA",Apr 2026,"1,702",38,5.8%,4.9%,5.9%,1.1%,1.5%,3.1%,-


## Analysis

The analysis was conducted on the top 200 of 713 metro regions by population that have time series covering at least five years. The top 200 metro areas are likely to face lower volatility in rent trends.

The top 5 metro areas by 1-year average rent growth include: 
1. Cedar Rapids, IA
2. Evansville, IN
3. Binghamton, NY
4. Reno, NV
5. Gulfport, MS

Cedar Rapids, IA experienced the fastest 1-year compound at 12.9%, followed closely by Evansville, IN (12.0%). These metro areas were home to approximately 280,000 and 240,000 residents, respectively. Cedar Rapids experienced the highest growth rate and most stable upward trend (declining standard deviation of rental growth rates). Evansville, IN experienced the second highest growth rate accompanied by a increasingly volatile trend (rising volatility of rental growth rates).


In [604]:

# Show regions with fastest growin rates
zori.loc[zori["Date"] == zori["Date"].max()].sort_values(
    ["CAGR_1yr", "CAGR_2yr", "CAGR_5yr"], ascending=False
).head(7).style.set_caption(
    """
    <div style='font-size:18px; font-weight:600;'>Which cities are leading rental rate growth?</div>
    <div style='font-size:13px; color:#888; margin-bottom: 2px;'>Rank movements over the last 18 months</div>
"""
).format(
    {
        "Date": lambda x: x.strftime("%b %Y"),
        "Zori": "{:,.0f}",
        "CAGR_1yr": "{:.1%}",
        "CAGR_2yr": "{:.1%}",
        "CAGR_5yr": "{:.1%}",
    }
).hide(
    axis="index"
)

RegionName,Date,Zori,SizeRank,CAGR_1yr,CAGR_2yr,CAGR_5yr,SD_1yr,SD_2yr,SD_5yr
"Lake Charles, LA",Apr 2026,"1,139",213,14.5%,9.6%,1.2%,0.029000,0.035000,0.052000
"Cedar Rapids, IA",Apr 2026,"1,014",179,12.9%,8.5%,6.2%,0.031000,0.042000,0.037000
"Evansville, IN",Apr 2026,"1,001",165,12.0%,9.7%,9.9%,0.085000,0.062000,0.053000
"Binghamton, NY",Apr 2026,"1,213",195,8.9%,7.1%,9.6%,0.058000,0.051000,0.058000
"Reno, NV",Apr 2026,"1,783",113,8.9%,5.9%,4.3%,0.033000,0.032000,0.036000
"Gulfport, MS",Apr 2026,"1,206",133,8.3%,6.6%,8.2%,0.041000,0.043000,0.049000
"Grand Forks, ND",Apr 2026,"1,090",377,7.6%,7.3%,5.7%,0.032000,0.026000,0.024000


In [605]:
# Show regions with slowest growin rates
zori.loc[zori["Date"] == zori["Date"].max()].sort_values(
    ["CAGR_1yr", "CAGR_2yr", "CAGR_5yr"], ascending=True
).head(7).style.set_caption(
    """
    <div style='font-size:18px; font-weight:600;'>Which cities are leading rental rate growth?</div>
    <div style='font-size:13px; color:#888; margin-bottom: 2px;'>Rank movements over the last 18 months</div>
"""
).format(
    {
        "Date": lambda x: x.strftime("%b %Y"),
        "Zori": "{:,.0f}",
        "CAGR_1yr": "{:.1%}",
        "CAGR_2yr": "{:.1%}",
        "CAGR_5yr": "{:.1%}",
    }
).hide(
    axis="index"
)

RegionName,Date,Zori,SizeRank,CAGR_1yr,CAGR_2yr,CAGR_5yr,SD_1yr,SD_2yr,SD_5yr
"Cape Coral, FL",Apr 2026,"1,664",78,-5.4%,-4.3%,4.4%,0.022000,0.018000,0.051000
"North Port, FL",Apr 2026,"1,824",73,-5.4%,-3.3%,4.9%,0.013000,0.014000,0.061000
"Austin, TX",Apr 2026,"1,448",29,-3.3%,-3.9%,1.0%,0.016000,0.017000,0.040000
"Moses Lake, WA",Apr 2026,"1,376",403,-3.2%,-1.1%,5.1%,0.028000,0.027000,0.033000
"Greeley, CO",Apr 2026,"1,458",161,-3.2%,-0.5%,2.8%,0.028000,0.024000,0.027000
"San Antonio, TX",Apr 2026,"1,239",24,-3.1%,-2.4%,1.4%,0.015000,0.018000,0.026000
"Savannah, GA",Apr 2026,"1,623",138,-2.9%,-0.3%,5.9%,0.014000,0.017000,0.039000


In [607]:
# print(zori.head())

import os

db_path = "storage/data.duckdb"

os.makedirs(os.path.dirname(db_path), exist_ok=True)
con = duckdb.connect(db_path)

con.execute("DROP TABLE IF EXISTS zori")
con.register("zori_df", zori)
con.execute("CREATE TABLE zori AS SELECT * FROM zori_df")

query = """
SELECT * FROM zori
ORDER BY Date, CAGR_1yr DESC
LIMIT 20 
"""

df = con.execute(query).df()
df


# with duckdb.con(db_path) as con:
#     df = con.execute(query).df()


,RegionName,Date,Zori,SizeRank,CAGR_1yr,CAGR_2yr,CAGR_5yr,SD_1yr,SD_2yr,SD_5yr
0,"Fresno, CA",2020-01-31,1026.0,57,0.106,0.096,0.070,0.024,0.024,0.023
1,"Prescott Valley, AZ",2020-01-31,1179.0,198,0.092,0.078,0.072,0.029,0.029,0.030
2,"Phoenix, AZ",2020-01-31,1222.0,11,0.081,0.084,0.079,0.012,0.014,0.018
3,"Portland, ME",2020-01-31,1421.0,105,0.079,0.051,0.048,0.049,0.050,0.044
4,"Tucson, AZ",2020-01-31,877.0,54,0.077,0.079,0.055,0.022,0.020,0.022
5,"Birmingham, AL",2020-01-31,1070.0,51,0.073,0.067,0.042,0.029,0.038,0.040
6,"Manchester, NH",2020-01-31,1345.0,131,0.070,0.067,0.055,0.030,0.033,0.035
7,"Ogden, UT",2020-01-31,1050.0,87,0.066,0.076,0.070,0.030,0.028,0.032
8,"Boise City, ID",2020-01-31,1079.0,79,0.064,0.077,0.068,0.022,0.022,0.022
9,"Charlottesville, VA",2020-01-31,1308.0,212,0.063,0.048,0.044,0.032,0.045,0.043


In [608]:
# import matplotlib.pyplot as plt

# zori_subset = zori[zori['Growth_Rank'] <= 10]
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(12,6))

# sns.lineplot(
#     data=zori.loc[(zori['Date'] > '2020') & (zori['Growth_Rank'] <= 5)],
#     x='Date',
#     y='Growth_Rank',
#     hue='RegionName',
#     linewidth=2
# )

# plt.gca().invert_yaxis()  # so Rank 1 is at the top
# plt.title("Growth Rank Over Time by Region")
# plt.xlabel("Date")
# plt.ylabel("Growth Rank (1 = Highest)")
# plt.legend(title="Region", bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

In [609]:
# target_date = "2023-12-31"

# top_regions = (
#     zori[zori['Date'] == target_date]
#     .nsmallest(5, 'Growth_Rank')['RegionName']
# )

# subset = zori[zori['RegionName'].isin(top_regions)]

# plt.figure(figsize=(12,6))
# sns.lineplot(
#     data=subset,
#     x='Date',
#     y='Growth_Rank',
#     hue='RegionName',
#     linewidth=2
# )
# plt.gca().invert_yaxis()
# plt.title("Top 5 Regions by Growth Rank")
# plt.show()



In [610]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from datetime import datetime


# state_avg = (
#     zori.groupby(['StateName', 'Date'])['Growth_Rank']
#         .mean()
#         .reset_index()
# )

# # print(state_avg.head())
# plt.figure(figsize=(12,6))
# sns.scatterplot(
#     data=state_avg.loc[state_avg['Growth_Rank'] <= 10],
#     x='Date',
#     y='Growth_Rank',
#     hue='StateName'
# )
# plt.gca().invert_yaxis()
# plt.title("Average Growth Rank by State")
# plt.show()

# target_date = "2023-12-31"

# top_regions = (
#     zori[zori['Date'] == target_date]
#     .nsmallest(5, 'Growth_Rank')['RegionName']
# )
# top_cities = (
#     zori[zori['Date'] == target_date]
#     .nsmallest(5, 'Growth_Rank')['RegionName']
# )



# print(top_cities.tail(10))
# subset = zori[zori['RegionName'].isin(top_regions)]

# subset = zori.loc[(zori['Date'] > '2023-01-01') & (zori['Growth_Rank'] <= 5)]

# plt.figure(figsize=(14,7))
# sns.scatterplot(data=subset, x='Date', y='Growth_Rank', hue='RegionName', s=80)
# sns.lineplot(data=subset, x='Date', y='Growth_Rank', hue='RegionName')



# plt.gca().invert_yaxis()

# for _, row in subset.iterrows():
#     plt.text(row['Date'], row['Growth_Rank'], row['RegionName'], fontsize=8)

# plt.show()





## Get table of growth ranks 
- (1 represents highest moving average growth rate for rental rates)

In [612]:
# import pandas as pd
# from dateutil.relativedelta import relativedelta

# def closest_prior_date(df, target_date):
#     dates = df['Date'].unique()
#     dates = dates[dates <= target_date]
#     if len(dates) == 0:
#         return None
#     return dates.max()

# latest_date = zori['Date'].max()

# # TOP_N = 300
# # top_regions = (
# #     zori[zori['Date'] == latest_date]
# #     .nsmallest(TOP_N, 'Growth_Rank')['RegionName']
# #     .tolist()
# # )

# panel = (
#     # zori[zori['RegionName'].isin(top_regions)]
#     zori
#     .pivot_table(
#         index='RegionName',
#         columns=['Date'],
#         values='Growth_Rank'
#     )
# )

# # print(panel.head())
# raw_dates = {
#     "current": latest_date,
#     "lag_6mo": latest_date - relativedelta(months=6),
#     "lag_12mo": latest_date - relativedelta(months=12),
#     "lag_18mo": latest_date - relativedelta(months=18),
#     # "lag_3yr": latest_date - relativedelta(years=3, months=6),
# }

# actual_dates = {}

# for key, dt in raw_dates.items():
#     actual_dates[key] = closest_prior_date(zori, dt)

# actual_dates

# panel_dates = panel.columns.intersection(actual_dates.values())
# panel = panel.loc[:, panel_dates]
# panel.columns = [
#     "Current Rank",
#     "Rank (-6 mo)",
#     "Rank (-12 mo)",
#     "Rank (-18 mo)",
# ]

# panel = panel.merge(zori[['RegionName', 'SizeRank']].drop_duplicates(), on='RegionName', how='left')

# panel.set_index('RegionName', inplace=True)
# panel.sort_values("Current Rank", inplace=True)

# # Apply a Size filter to focus on larger markets and avoid noise from small regions with volatile ranks
# panel = panel[[c for c in panel.columns if c != 'SizeRank']]
# panel["Avg Rank"] = panel.mean(axis=1)
# # panel["City"] = panel.index.to_series().str.extract(r'^\s*([A-Za-z ]+),')[0].values
# # panel["State"] = panel.index.to_series().str.extract(r',\s*([A-Z]{2})$')[0].values
# panel.sort_values('Current Rank', inplace=True)
# panel = panel.merge(zori[['RegionName', 'SizeRank']].drop_duplicates(), on='RegionName', how='left')
# # panel.head(20)
# # panel.style.format("{:.0f}")


# panel.head(20).style.set_caption(
#     """
#     <div style='font-size:18px; font-weight:600;'>Which cities are leading rental rate growth?</div>
#     <div style='font-size:13px; color:#888; margin-bottom: 2px;'>Rank trajectories over the last 18 months</div>
#     """
# ).set_table_styles([
#     {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left")]}
# ]).format({
#     "Rank (-18 mo)": "{:.0f}",
#     "Rank (-12 mo)": "{:.0f}",
#     "Rank (-6 mo)": "{:.0f}",
#     "Current Rank": "{:.0f}",
#     "Avg Rank": "{:.0f}"
# }).hide(axis="index")